In [ ]:
import os
import json
import torch
import requests
import zipfile
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
from PIL import Image
from tqdm import tqdm
from scipy.linalg import eigh
from datasets import load_dataset
from transformers import BlipProcessor, BlipForImageTextRetrieval

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
GRID_SIZE = 24

def setup_coco_dataset():
    annotation_file = "instances_val2017.json"
    
    if not os.path.exists(annotation_file):
        print("Downloading MS-COCO validation bounding boxes...")
        url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
        r = requests.get(url, stream=True)
        with open("annotations.zip", 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        with zipfile.ZipFile("annotations.zip", 'r') as zip_ref:
            zip_ref.extract("annotations/instances_val2017.json", ".")
        os.rename("annotations/instances_val2017.json", annotation_file)
        os.remove("annotations.zip")
        print("Download complete.")

    with open(annotation_file, 'r') as f:
        coco_data = json.load(f)

    categories = {cat['id']: cat['name'] for cat in coco_data['categories']}
    image_to_objects = {}
    
    for ann in coco_data['annotations']:
        img_id = ann['image_id']
        if img_id not in image_to_objects:
            image_to_objects[img_id] = []
        image_to_objects[img_id].append({
            'category_name': categories[ann['category_id']],
            'bbox': ann['bbox']
        })

    dataset = load_dataset("phiyodr/coco2017", split="validation", streaming=True)
    return dataset, image_to_objects

coco_dataset, coco_objects = setup_coco_dataset()
print(f"MS-COCO Dataset Loaded: {len(coco_objects)} annotated images found.")

In [ ]:
def extract_all_heatmaps(model, inputs, valid_indices, tgt_rel):
    heatmaps = {}
    attn_maps = []
    hooks = []

    def hook_fn(module, inp, out):
        attn_maps.append(inp[0].detach().cpu())

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks.append(module.register_forward_hook(hook_fn))

    with torch.no_grad():
        model(**inputs)

    for h in hooks:
        h.remove()

    all_layers_attn = torch.stack(attn_maps)

    # 1. Raw Attention
    avg_attn = all_layers_attn.mean(dim=(0, 1, 2))[valid_indices, 1:]
    hm_raw = avg_attn[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_raw = hm_raw / (hm_raw.max() + 1e-10)
    heatmaps["Raw Attention"] = hm_raw.numpy()

    # 2. Attention Rollout
    T, I = all_layers_attn.shape[-2], all_layers_attn.shape[-1]
    rollout = torch.eye(T + I)

    for layer_attn in all_layers_attn:
        avg_heads = layer_attn.mean(dim=(0, 1))
        full = torch.zeros((T + I, T + I))
        full[:T, T:] = avg_heads
        full[T:, :T] = avg_heads.T
        full = 0.5 * full + 0.5 * torch.eye(T + I)
        rollout = rollout @ full

    text_to_img_rollout = rollout[:T, T:]
    filtered_rollout = text_to_img_rollout[valid_indices, 1:]
    hm_roll = filtered_rollout[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_roll = hm_roll / (hm_roll.max() + 1e-10)
    heatmaps["Attention Rollout"] = hm_roll.numpy()

    # 3. Deep-Spec (Ours)
    A_global = all_layers_attn.mean(dim=(0, 1, 2))[valid_indices, 1:]
    A_global = A_global / (A_global.max() + 1e-10)
    W_c = A_global.numpy()

    T_ds, I_ds = W_c.shape
    N = T_ds + I_ds
    W = np.zeros((N, N))
    W[:T_ds, T_ds:] = W_c
    W[T_ds:, :T_ds] = W_c.T
    W += 1e-5  # Minimal connectivity regularizer

    D = np.sum(W, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
    L_sym = np.eye(N) - D_inv_sqrt @ W @ D_inv_sqrt
    _, eigvecs = eigh(L_sym)

    k = 5
    V_text = eigvecs[:T_ds, 1:k+1]
    V_img = eigvecs[T_ds:, 1:k+1]
    denoised_affinity = V_text @ V_img.T

    target_affinity = np.abs(denoised_affinity[tgt_rel, :])
    hm_ds = target_affinity.max(axis=0).reshape(GRID_SIZE, GRID_SIZE)
    hm_ds = (hm_ds - hm_ds.min()) / (hm_ds.max() - hm_ds.min() + 1e-10)
    heatmaps["Deep-Spec (Ours)"] = hm_ds

    # 4. Gradient Extractions (ViT-Grad-CAM & Chefer)
    activations, gradients = None, None
    attn_maps_grad, grad_maps = [], []

    def fwd_gcam(m, inp, out): nonlocal activations; activations = out[0].detach()
    def bwd_gcam(m, gin, gout): nonlocal gradients; gradients = gout[0].detach()
    
    def fwd_chefer(m, inp, out):
        attn = inp[0]
        attn_maps_grad.append(attn)
        attn.retain_grad()
        attn.register_hook(lambda g: grad_maps.append(g))

    hooks_grad = []
    target_layer = model.vision_model.encoder.layers[-1]
    hooks_grad.append(target_layer.register_forward_hook(fwd_gcam))
    hooks_grad.append(target_layer.register_full_backward_hook(bwd_gcam))

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks_grad.append(module.register_forward_hook(fwd_chefer))

    model.zero_grad()
    outputs = model(**inputs)
    itm_score = outputs.itm_score[0, 1]
    itm_score.backward()

    for h in hooks_grad: h.remove()

    if gradients is not None and activations is not None:
        weights = gradients.mean(dim=(0, 1))
        cam = (weights * activations).sum(dim=-1).squeeze(0)
        cam = torch.relu(cam)[1:]
        cam = cam.reshape(GRID_SIZE, GRID_SIZE)
        cam = cam / (cam.max() + 1e-10)
        heatmaps["ViT-Grad-CAM"] = cam.detach().cpu().numpy()
    else:
        heatmaps["ViT-Grad-CAM"] = np.zeros((GRID_SIZE, GRID_SIZE))

    if len(attn_maps_grad) > 0 and len(grad_maps) > 0:
        hm_chefer = torch.zeros_like(attn_maps_grad[0].mean(dim=(0, 1)))
        for attn, grad in zip(attn_maps_grad, grad_maps):
            hm_chefer += (attn * grad).mean(dim=(0, 1))

        hm_chefer = hm_chefer / len(attn_maps_grad)
        hm_chefer = hm_chefer[valid_indices, 1:]
        hm_chefer = torch.clamp(hm_chefer, min=0)

        hm_chefer_final = hm_chefer[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
        hm_chefer_final = hm_chefer_final / (hm_chefer_final.max() + 1e-10)
        heatmaps["Attn x Grad (Chefer)"] = hm_chefer_final.detach().cpu().numpy()
    else:
        heatmaps["Attn x Grad (Chefer)"] = np.zeros((GRID_SIZE, GRID_SIZE))

    return heatmaps

In [ ]:
def run_pointing_game_benchmark(model_name, num_evals=1000):
    print(f"\nLoading Model: {model_name}...")
    processor = BlipProcessor.from_pretrained(model_name)
    model = BlipForImageTextRetrieval.from_pretrained(model_name).to(device)
    model.eval()

    methods = ["Raw Attention", "Attention Rollout", "ViT-Grad-CAM", "Attn x Grad (Chefer)", "Deep-Spec (Ours)"]
    hits = {m: 0 for m in methods}
    total_evaluated = 0
    iterator = iter(coco_dataset)

    print(f"Starting Benchmark over {num_evals} MS-COCO Validation objects...\n")
    with tqdm(total=num_evals, desc="Evaluating", unit="obj") as pbar:
        while total_evaluated < num_evals:
            try:
                data = next(iterator)
                image_id = data.get('image_id')

                if image_id not in coco_objects: continue
                objects = coco_objects[image_id]

                if 'image' in data and data['image'] is not None:
                    image = data['image'].convert("RGB")
                elif 'coco_url' in data:
                    response = requests.get(data['coco_url'], timeout=10)
                    image = Image.open(BytesIO(response.content)).convert("RGB")
                else: continue

                orig_W, orig_H = image.size
                caption = data['captions'][0] if isinstance(data.get('captions'), list) else data.get('text', "")
                if not caption: continue

                target_noun = next((obj['category_name'] for obj in objects if obj['category_name'].lower() in caption.lower()), None)
                if not target_noun: continue
                
                all_target_bboxes = [obj['bbox'] for obj in objects if obj['category_name'] == target_noun]

                inputs = processor(images=image, text=caption, return_tensors="pt").to(device)
                tokens = processor.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
                special_ids = processor.tokenizer.all_special_ids
                valid_indices = [i for i, idx in enumerate(inputs["input_ids"][0].tolist()) if idx not in special_ids]

                target_indices = [i for i, t in enumerate(tokens) if i in valid_indices and any(w in t.lower().replace("##", "") for w in target_noun.lower().split())]
                if not target_indices: continue

                tgt_rel = [valid_indices.index(idx) for idx in target_indices]
                heatmaps = extract_all_heatmaps(model, inputs, valid_indices, tgt_rel)
                
                if heatmaps is None: continue

                for m_name in methods:
                    hm = heatmaps[m_name]
                    if hm.max() == 0: continue

                    max_idx = np.argmax(hm)
                    max_y_patch, max_x_patch = np.unravel_index(max_idx, hm.shape)

                    pixel_x = ((max_x_patch + 0.5) / GRID_SIZE) * orig_W
                    pixel_y = ((max_y_patch + 0.5) / GRID_SIZE) * orig_H

                    hit = any(x_min <= pixel_x <= (x_min + w) and y_min <= pixel_y <= (y_min + h) for (x_min, y_min, w, h) in all_target_bboxes)
                    if hit: hits[m_name] += 1

                total_evaluated += 1
                pbar.update(1)

            except StopIteration: break
            except Exception: continue

    print("\n" + "=" * 65)
    print(f" FINAL POINTING GAME ACCURACY ({model_name}) ".center(65))
    print("=" * 65)
    for m_name in methods:
        acc = (hits[m_name] / total_evaluated) * 100 if total_evaluated > 0 else 0
        prefix = ">> " if "Deep-Spec" in m_name else "   "
        print(f"{prefix}{m_name:<25} | {acc:.2f}%")
    print("=" * 65)
    
    return model, processor # Return for visualization

In [ ]:

# ==============================================================================

# Run BLIP-Base
base_model, base_processor = run_pointing_game_benchmark("Salesforce/blip-itm-base-coco", num_evals=1000)

# Run BLIP-Large (Optional: uncomment to run large variant)
# large_model, large_processor = run_pointing_game_benchmark("Salesforce/blip-itm-large-coco", num_evals=1000)